In [1]:
import pandas as pd
import numpy as np
import pathlib

In [2]:
countries = ["nigeria", "india"]
vehicles = {
    "nigeria": "bouillon",
    "india": "rice",
}

data_needs = {
    "Country-Vehicle": {
        "Vehicle consumption by WRA -- any": "vehicle_consumption/any",
        'Vehicle "fortifiability" (essentially amount industrially produced)': "vehicle_consumption/fortifiability",
        "Vehicle consumption by WRA -- amount": "vehicle_consumption/amount",
    },
    "Country-Vehicle-Fort": {
        "Vehicle fortification at baseline -- any": "baseline_fortification/any_coverage",
        "Vehicle fortification at baseline -- amount among fortified": "baseline_fortification/concentration",
    },
    "Scenario Definition": {
        "Intervention coverage % of fortifiable and unfortified": "intervention_fortification/any_coverage",
        "Intervention effective % of newly fortified": "intervention_fortification/effective_coverage",
        "Vehicle fortification in intervention -- amount among fortified": "intervention_fortification/concentration",
    },
}

data_point_names = {
    "mean": "mean",
    "standard deviation": "sd",
}

In [3]:
pregnancy_sim_data_dir = (
    "../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data/"
)
pathlib.Path(pregnancy_sim_data_dir).mkdir(parents=True, exist_ok=True)

In [4]:
def check_with_total(rows):
    rows = rows.set_index("wealth_quintile").value
    total = rows.loc["all"]
    by_quintile = rows[rows.index != "all"]
    assert np.isclose(by_quintile.mean(), total, atol=0, rtol=0.1)

In [5]:
for country in countries:
    print(f"Processing {country}")
    for sheet_name, sheet_data_needs in data_needs.items():
        sheet = pd.read_excel(
            "./Data Extraction Sheet.xlsx", sheet_name=f"{sheet_name} Extraction"
        )
        sheet = sheet[sheet.Country.str.lower() == country]

        if "Fortificant" in sheet.columns:
            sheet = sheet[sheet.Fortificant == "Iron"]

        if "Vehicle" in sheet.columns:
            assert (sheet.Vehicle.str.lower() == vehicles[country]).all()

        for need, short_need_name in sheet_data_needs.items():
            if need not in sheet["Data need"].values:
                # Needs filled by microdata
                print(f"Need {need} not extracted")
                continue

            print(f"Data need: {need}")
            need_rows = sheet[sheet["Data need"] == need]

            if country == "india" and short_need_name == "vehicle_consumption/fortifiability":
                # This is a special case with more processing to harmonize different source of information;
                # see the next cell 
                continue

            if "Quintile" in need_rows.columns:
                assert set(
                    need_rows["Quintile"]
                    == {"All", "Lowest", "Second", "Middle", "Fourth", "Highest"}
                )
                definition_columns = ["Data point name", "Quintile"]
                by_quintile = True
            else:
                definition_columns = ["Data point name"]
                by_quintile = False

            assert (need_rows.groupby(definition_columns).size() == 1).all()

            if (
                need.endswith("-- any")
                or short_need_name.endswith("_coverage")
                or short_need_name == "vehicle_consumption/fortifiability"
            ):
                # Percentage
                assert (need_rows.Units == "%").all()
                assert (need_rows["Data point name"] == "percentage").all()
            elif need.endswith("-- amount") or short_need_name.endswith("amount"):
                # Consumption in g/day
                assert (need_rows.Units == "g/day").all()
                assert (
                    need_rows["Data point name"].isin(["mean", "standard deviation"])
                ).all()
            elif need.endswith("-- amount among fortified") or short_need_name.endswith(
                "concentration"
            ):
                # Concentration in mcg/g
                assert (need_rows.Units == "mcg/g").all()
                assert (need_rows["Data point name"] == "concentration").all()
            else:
                raise ValueError()

            data_points = need_rows["Data point name"].unique()
            for data_point in data_points:
                data_point_rows = need_rows[need_rows["Data point name"] == data_point]
                data_point_rows = data_point_rows[
                    [
                        c
                        for c in data_point_rows
                        if c in ["Vehicle", "Quintile", "Value"]
                    ]
                ].rename(
                    columns={
                        "Vehicle": "vehicle_name",
                        "Quintile": "wealth_quintile",
                        "Value": "value",
                    }
                )
                data_point_rows["vehicle_name"] = (
                    data_point_rows.vehicle_name.str.lower()
                )

                if by_quintile:
                    data_point_rows["wealth_quintile"] = (
                        data_point_rows.wealth_quintile.str.lower()
                    )
                    if country == "india" and short_need_name == "vehicle_consumption/fortifiability":
                        data_point_rows = rescale_to_total(data_point_rows)
                    check_with_total(data_point_rows)
                    data_point_rows = data_point_rows[
                        data_point_rows.wealth_quintile != "all"
                    ]

                if len(data_points) == 1:
                    dir_name = short_need_name
                else:
                    dir_name = f"{short_need_name}/{data_point_names[data_point]}"

                file_path = f"{pregnancy_sim_data_dir}/{dir_name}/{country}.csv"
                pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
                data_point_rows.to_csv(file_path, index=False)

                if short_need_name == "baseline_fortification/any_coverage":
                    # NOTE: For extractions, any == full coverage! We assume people are either fully
                    # covered or not. This is probably reasonable in Nigeria -- why would people
                    # buy multiple different types of bouillon?
                    # We do something more sophisticated with India from microdata -- especially
                    # relevant because lots of people have ration cards for a certain amount from one source.
                    dir_name = "baseline_fortification/full_coverage"
                    file_path = f"{pregnancy_sim_data_dir}/{dir_name}/{country}.csv"
                    pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
                    data_point_rows.to_csv(file_path, index=False)

Processing nigeria


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Data need: Vehicle consumption by WRA -- any
Data need: Vehicle "fortifiability" (essentially amount industrially produced)
Data need: Vehicle consumption by WRA -- amount
Data need: Vehicle fortification at baseline -- any
Data need: Vehicle fortification at baseline -- amount among fortified
Data need: Intervention coverage % of fortifiable and unfortified
Data need: Intervention effective % of newly fortified
Data need: Vehicle fortification in intervention -- amount among fortified
Processing india


Need Vehicle consumption by WRA -- any not extracted
Data need: Vehicle "fortifiability" (essentially amount industrially produced)
Need Vehicle consumption by WRA -- amount not extracted
Need Vehicle fortification at baseline -- any not extracted
Data need: Vehicle fortification at baseline -- amount among fortified
Data need: Intervention coverage % of fortifiable and unfortified
Data need: Intervention effective % of newly fortified
Data need: Vehicle fortification in intervention -- amount among fortified


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [6]:
# We apply the India industry consolidation (overall fortifiability from the extraction sheet) to the rice *not
# distributed by the government*.
# First we disaggregate the industry consolidation number by wealth according to a proxy from HCES: the proportion purchased.

sheet = pd.read_excel(
    "./Data Extraction Sheet.xlsx", sheet_name="Country-Vehicle Extraction"
)
sheet = sheet[sheet.Country.str.lower() == "india"]
assert (sheet.Vehicle.str.lower() == vehicles["india"]).all()

overall_fortifiability_row = sheet[sheet["Data need"] == 'Vehicle "fortifiability" (essentially amount industrially produced)']
assert len(overall_fortifiability_row) == 1
assert (overall_fortifiability_row.Quintile == "All").all()

industry_consolidation = float(overall_fortifiability_row.Value.iloc[0])
industry_consolidation

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


0.513

In [7]:
disparity_proxy = pd.read_csv('../hces/india_fortifiability_disparities.csv').set_index("wealth_quintile").value
disparity_proxy

wealth_quintile
lowest     0.895527
second     0.911167
middle     0.935564
fourth     0.945074
highest    0.966887
Name: value, dtype: float64

In [8]:
# Equally weighting the quintiles, which isn't quite right but close
industry_consolidation_by_quintile = (industry_consolidation * disparity_proxy) / disparity_proxy.mean()
industry_consolidation_by_quintile

wealth_quintile
lowest     0.493536
second     0.502156
middle     0.515601
fourth     0.520842
highest    0.532864
Name: value, dtype: float64

In [9]:
government_rice = pd.read_csv('../hces/india_proportion_government.csv').set_index("wealth_quintile").value
government_rice

wealth_quintile
lowest     0.574839
second     0.582899
middle     0.527186
fourth     0.452597
highest    0.251798
Name: value, dtype: float64

In [10]:
# We assume all rice distributed by the government is fortifiable.
# The industry consolidation applies to non-government distributed rice.
fortifiability_by_quintile = (
    government_rice + (1 - government_rice) * industry_consolidation_by_quintile
)
fortifiability_by_quintile

wealth_quintile
lowest     0.784671
second     0.792349
middle     0.770970
fourth     0.737707
highest    0.650488
Name: value, dtype: float64

In [11]:
file_path = f"{pregnancy_sim_data_dir}/vehicle_consumption/fortifiability/india.csv"
pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
fortifiability_by_quintile.reset_index().assign(vehicle_name="rice").to_csv(file_path, index=False)